In [20]:
# Deep learning for joint OD word-bit prediction
# We build a multi-label target over 16 words x 32 bits = 512 outputs.
# Each output target y[w*32 + b] corresponds to whether bit b of CipherTextDiff_w is 1.
# The model learns P(OD_word=w AND OD_bit=b | (ID_word, ID_bit, plaintext context)).

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ---------------------------
# 1) Prepare features and labels
# ---------------------------

# Encode input word symbol (e.g., 'c','d','e','f') to integer id
le_word_joint = LabelEncoder()
unique_words = dataset['IDWordIndex'].astype(str)
le_word_joint.fit(unique_words)

# Utility: hex string (up to 32-bit) -> 32-bit binary array
def hex_to_bits32(x: str) -> np.ndarray:
    # Ensure string, convert hex->int->binary string->pad->array of ints
    s = format(int(str(x), 16), '032b')
    return np.fromiter((1 if ch=='1' else 0 for ch in s), dtype=np.uint8, count=32)

# Build X features: [input_word_id, input_bit, (optional) lightweight PT context]
# Keep it simple/fast first: only input_word_id and input_bit. You can append PT bytes later if needed.
X_list = []
Y_list = []

# Iterate rows once; for each row create a 512-dim label
for _, row in dataset.iterrows():
    input_word_id = le_word_joint.transform([str(row['IDWordIndex'])])[0]
    input_bit = int(row['IDBitIndex'])

    # Feature vector: [input_word_id, input_bit]
    X_list.append([input_word_id, input_bit])

    # Build 512 labels by concatenating bits of each CipherTextDiff_w (w=0..15)
    bits_all = []
    for w in range(16):
        bits = hex_to_bits32(row[f'CipherTextDiff_{w}'])  # shape (32,)
        bits_all.append(bits)
    y = np.concatenate(bits_all, axis=0)  # shape (512,)
    Y_list.append(y)

X_np = np.asarray(X_list, dtype=np.int64)            # shape (N, 2)
Y_np = np.asarray(Y_list, dtype=np.uint8)           # shape (N, 512)

# Standardize numeric features for DL stability
scaler_joint = StandardScaler()
X_np_scaled = scaler_joint.fit_transform(X_np.astype(np.float32))

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(X_np_scaled, Y_np, test_size=0.2, random_state=42, stratify=Y_np.sum(axis=1) > 0)

# ---------------------------
# 2) Torch dataset/dataloader
# ---------------------------
class ODJointDataset(Dataset):
    def __init__(self, X: np.ndarray, Y: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).float()
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

batch_size = 1024
train_ds = ODJointDataset(X_train, y_train)
val_ds = ODJointDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

# ---------------------------
# 3) Define MLP for 512 outputs
# ---------------------------
class ODJointMLP(nn.Module):
    def __init__(self, in_dim: int = 2, hidden: int = 256, out_dim: int = 512, p_drop: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x):
        return self.net(x)  # logits for 512 labels

# Instantiate model, loss (BCE-with-logits), optimizer
model_joint = ODJointMLP(in_dim=X_train.shape[1], hidden=256, out_dim=512, p_drop=0.2)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_joint.parameters(), lr=1e-3)

# ---------------------------
# 4) Training loop with validation
# ---------------------------

def evaluate(model: nn.Module, data_loader: DataLoader):
    model.eval()
    total_loss = 0.0
    y_true_all = []
    y_prob_all = []
    with torch.no_grad():
        for xb, yb in data_loader:
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            probs = torch.sigmoid(logits).cpu().numpy()
            y_prob_all.append(probs)
            y_true_all.append(yb.cpu().numpy())
    y_true = np.concatenate(y_true_all, axis=0)
    y_prob = np.concatenate(y_prob_all, axis=0)
    # Macro AUPRC/AUROC across 512 outputs
    try:
        ap = average_precision_score(y_true, y_prob, average='macro')
    except Exception:
        ap = np.nan
    try:
        auroc = roc_auc_score(y_true, y_prob, average='macro')
    except Exception:
        auroc = np.nan
    avg_loss = total_loss / len(data_loader.dataset)
    return avg_loss, ap, auroc

epochs = 10  # adjust upward if needed
best_val_ap = -1.0
best_state_dict = None

for epoch in range(1, epochs+1):
    model_joint.train()
    running = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model_joint(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * xb.size(0)
    train_loss = running / len(train_loader.dataset)
    val_loss, val_ap, val_auc = evaluate(model_joint, val_loader)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_mAP={val_ap:.4f} val_AUROC={val_auc:.4f}")
    # Track best by validation mAP
    if (not np.isnan(val_ap)) and val_ap > best_val_ap:
        best_val_ap = val_ap
        best_state_dict = {k: v.cpu().clone() for k, v in model_joint.state_dict().items()}

# Load best weights if improved
if best_state_dict is not None:
    model_joint.load_state_dict(best_state_dict)

# ---------------------------
# 5) Prediction utilities
# ---------------------------

def predict_top_k_od(input_word: str, input_bit: int, top_k: int = 10):
    """
    Return top-k OD (word, bit) pairs with highest predicted probability.
    """
    x = np.asarray([[le_word_joint.transform([str(input_word)])[0], int(input_bit)]], dtype=np.int64)
    x = scaler_joint.transform(x.astype(np.float32))
    with torch.no_grad():
        logits = model_joint(torch.from_numpy(x).float())
        probs = torch.sigmoid(logits).cpu().numpy()[0]  # shape (512,)
    # Decode indices -> (word, bit)
    idxs = np.argsort(-probs)[:top_k]
    results = []
    for idx in idxs:
        od_word = idx // 32
        od_bit = idx % 32
        results.append((int(od_word), int(od_bit), float(probs[idx])))
    return results

# Example usage (prints top-5 OD pairs)
print("\nExample: top-5 OD pairs for ('c', 0)")
for w, b, p in predict_top_k_od('c', 0, top_k=5):
    print(f"  OD_word={w:2d}, OD_bit={b:2d}, prob={p:.4f}")



Epoch 01 | train_loss=0.1944 val_loss=0.1709 val_mAP=0.1360 val_AUROC=nan
Epoch 02 | train_loss=0.1711 val_loss=0.1696 val_mAP=0.1400 val_AUROC=nan
Epoch 03 | train_loss=0.1698 val_loss=0.1684 val_mAP=0.1439 val_AUROC=nan
Epoch 04 | train_loss=0.1688 val_loss=0.1674 val_mAP=0.1470 val_AUROC=nan
Epoch 05 | train_loss=0.1681 val_loss=0.1665 val_mAP=0.1498 val_AUROC=nan
Epoch 06 | train_loss=0.1673 val_loss=0.1656 val_mAP=0.1514 val_AUROC=nan
Epoch 07 | train_loss=0.1667 val_loss=0.1650 val_mAP=0.1521 val_AUROC=nan
Epoch 08 | train_loss=0.1662 val_loss=0.1646 val_mAP=0.1525 val_AUROC=nan
Epoch 09 | train_loss=0.1659 val_loss=0.1643 val_mAP=0.1528 val_AUROC=nan
Epoch 10 | train_loss=0.1656 val_loss=0.1641 val_mAP=0.1531 val_AUROC=nan

Example: top-5 OD pairs for ('c', 0)
  OD_word= 0, OD_bit=25, prob=0.9083
  OD_word= 1, OD_bit=31, prob=0.8129
  OD_word= 1, OD_bit=26, prob=0.8109
  OD_word=13, OD_bit=25, prob=0.7840
  OD_word=10, OD_bit=31, prob=0.7767


In [21]:
# Evaluate model on validation set with concise summaries
val_loss, val_ap, val_auc = evaluate(model_joint, val_loader)
print(f"Validation: loss={val_loss:.4f}, mAP={val_ap:.4f}, AUROC={val_auc:.4f}")

# Quick sanity check across a few inputs
examples = [('c', 0), ('d', 5), ('e', 15)]
for iw, ib in examples:
    top = predict_top_k_od(iw, ib, top_k=5)
    print(f"\nInput ({iw}, {ib}) -> top-5 OD (word, bit, prob):")
    for w, b, p in top:
        print(f"  ({w:2d}, {b:2d}) p={p:.4f}")


Validation: loss=0.1641, mAP=0.1531, AUROC=nan

Input (c, 0) -> top-5 OD (word, bit, prob):
  ( 0, 25) p=0.9083
  ( 1, 31) p=0.8129
  ( 1, 26) p=0.8109
  (13, 25) p=0.7840
  (10, 31) p=0.7767

Input (d, 5) -> top-5 OD (word, bit, prob):
  ( 2, 26) p=0.8221
  ( 3, 30) p=0.8123
  ( 1, 28) p=0.7972
  (14, 28) p=0.7832
  (12, 29) p=0.7706

Input (e, 15) -> top-5 OD (word, bit, prob):
  ( 1, 26) p=0.9551
  (12, 24) p=0.8793
  (12, 27) p=0.8786
  ( 2, 24) p=0.8425
  ( 0, 30) p=0.8398


In [22]:
# Save model and encoders for later reuse
import joblib

save_bundle = {
    'state_dict': model_joint.state_dict(),
    'scaler': scaler_joint,
    'label_encoder_word': le_word_joint,
    'input_dim': X_train.shape[1],
    'hidden': 256,
    'output_dim': 512,
}

# Torch model is saved as state_dict alongside scikit objects
joblib.dump(save_bundle, 'joint_od_predictor_bundle.pkl')
print("Saved: joint_od_predictor_bundle.pkl")

# Usage note:
# To load later:
#   bundle = joblib.load('joint_od_predictor_bundle.pkl')
#   model = ODJointMLP(in_dim=bundle['input_dim'], hidden=bundle['hidden'], out_dim=bundle['output_dim'])
#   model.load_state_dict(bundle['state_dict'])
#   scaler = bundle['scaler']
#   le_word = bundle['label_encoder_word']
#   # Then call predict_top_k_od after recreating the function with these objects.



Saved: joint_od_predictor_bundle.pkl


In [23]:
# DL-only next-round prediction utilities
# We use the trained ODJointMLP to compute probabilities for all 16x32 OD bits,
# then aggregate to rank output words and bits.

import numpy as np

def predict_od_matrix(input_word: str, input_bit: int) -> np.ndarray:
    """
    Return a (16, 32) matrix of P(OD_word=w, OD_bit=b | input) from DL model.
    """
    x = np.asarray([[le_word_joint.transform([str(input_word)])[0], int(input_bit)]], dtype=np.int64)
    x = scaler_joint.transform(x.astype(np.float32))
    with torch.no_grad():
        logits = model_joint(torch.from_numpy(x).float())
        probs = torch.sigmoid(logits).cpu().numpy()[0]  # shape (512,)
    return probs.reshape(16, 32)

def rank_output_words(prob_matrix: np.ndarray, agg: str = 'sum'):
    """
    Rank output words by aggregated bit probability.
    agg in {'sum','max','mean'} determines how to aggregate across 32 bits.
    Returns list of tuples (word, score).
    """
    if agg == 'sum':
        scores = prob_matrix.sum(axis=1)
    elif agg == 'max':
        scores = prob_matrix.max(axis=1)
    elif agg == 'mean':
        scores = prob_matrix.mean(axis=1)
    else:
        raise ValueError("agg must be one of {'sum','max','mean'}")
    ranking = sorted([(w, float(scores[w])) for w in range(16)], key=lambda x: x[1], reverse=True)
    return ranking

def rank_output_bits(prob_matrix: np.ndarray, output_word: int, top_k: int = 8):
    """
    Rank bits within a chosen output word. Returns top_k list of (bit, prob).
    """
    row = prob_matrix[output_word]
    idxs = np.argsort(-row)[:top_k]
    return [(int(b), float(row[b])) for b in idxs]

def predict_next_round_dl(input_word: str, input_bit: int, top_k_words: int = 5, bit_top_k: int = 8, agg: str = 'sum'):
    """
    DL-only interface to get best output words and bits.
    Returns dict with:
      - prob_matrix: (16,32) numpy array
      - top_words: list[(word, score)] of length top_k_words
      - top_bits_per_word: dict[word] -> list[(bit, prob)] of length bit_top_k
    """
    pm = predict_od_matrix(input_word, input_bit)
    words_ranked = rank_output_words(pm, agg=agg)[:top_k_words]
    bits_per = {w: rank_output_bits(pm, w, top_k=bit_top_k) for w, _ in words_ranked}
    return {
        'prob_matrix': pm,
        'top_words': words_ranked,
        'top_bits_per_word': bits_per,
    }



In [24]:
# DL-only example usage
# Note: Ensure you've trained the DL model cells above first.

examples = [('c', 0), ('d', 5), ('e', 15)]
for iw, ib in examples:
    res = predict_next_round_dl(iw, ib, top_k_words=3, bit_top_k=5, agg='sum')
    print(f"\nInput ({iw}, {ib})")
    print("Top output words (agg=sum):")
    for w, s in res['top_words']:
        print(f"  word={w:2d} score={s:.4f}")
    for w, _ in res['top_words']:
        print(f"  Best bits in word {w}:")
        for b, p in res['top_bits_per_word'][w]:
            print(f"    bit={b:2d} p={p:.4f}")




Input (c, 0)
Top output words (agg=sum):
  word= 2 score=4.4380
  word=13 score=4.3819
  word=10 score=4.3669
  Best bits in word 2:
    bit=27 p=0.7714
    bit=30 p=0.7168
    bit=24 p=0.6444
    bit=26 p=0.5482
    bit=29 p=0.4948
  Best bits in word 13:
    bit=25 p=0.7840
    bit=28 p=0.7035
    bit=30 p=0.6375
    bit=24 p=0.5175
    bit=27 p=0.5135
  Best bits in word 10:
    bit=31 p=0.7767
    bit=28 p=0.6140
    bit=26 p=0.6005
    bit=25 p=0.5252
    bit=30 p=0.5091

Input (d, 5)
Top output words (agg=sum):
  word= 3 score=4.3352
  word=14 score=4.3184
  word= 6 score=4.2378
  Best bits in word 3:
    bit=30 p=0.8123
    bit=27 p=0.6256
    bit=25 p=0.6031
    bit=29 p=0.5131
    bit=26 p=0.4992
  Best bits in word 14:
    bit=28 p=0.7832
    bit=25 p=0.6274
    bit=31 p=0.6044
    bit=24 p=0.5069
    bit=30 p=0.4924
  Best bits in word 6:
    bit=26 p=0.5671
    bit=25 p=0.5503
    bit=29 p=0.5474
    bit=30 p=0.5414
    bit=28 p=0.5381

Input (e, 15)
Top output words (agg=